# Notebook 1：数据探索与可视化

**项目**：P2PNet-Soy 大豆种子定位与计数复现  
**论文**：Zhao et al., Plant Phenomics, 2022  
**DOI**：https://doi.org/10.34133/plantphenomics.0026

本 Notebook 完成以下内容：
1. 数据集基本统计（图像数量、尺寸分布）
2. 标注文件解析（点坐标读取）
3. 大豆种子计数分布分析
4. 图像与标注点的可视化展示

## 1. 环境准备

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体（如无需中文可注释）
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

print('环境加载完成 ✓')

## 2. 数据路径配置

> ⚠️ 请根据你本地的数据存放位置修改 `DATA_ROOT`

In [ ]:
# ===== 修改为你的数据路径 =====
DATA_ROOT = Path('../data/Soybean_seed_counting')
# ==============================

IMG_A = DATA_ROOT / 'images_a'
IMG_B = DATA_ROOT / 'images_b'
LBL_A = DATA_ROOT / 'labels_a'
LBL_B = DATA_ROOT / 'labels_b'
INDEX_A = DATA_ROOT / 'soybean_seed_counting_a.txt'
INDEX_B = DATA_ROOT / 'soybean_seed_counting_b.txt'

# 验证路径存在
for p in [IMG_A, IMG_B, LBL_A, LBL_B]:
    status = '✓' if p.exists() else '✗ 路径不存在，请检查！'
    print(f'{p.name}: {status}')

## 3. 数据集统计

论文中使用 24 个大豆品系，A面图像用于训练，B面图像用于测试（反之亦然）。

In [ ]:
# 统计图像数量
imgs_a = sorted(list(IMG_A.glob('*.png')))
imgs_b = sorted(list(IMG_B.glob('*.png')))
lbls_a = sorted(list(LBL_A.glob('*.txt')))
lbls_b = sorted(list(LBL_B.glob('*.txt')))

print(f'训练集 (A面)：图像 {len(imgs_a)} 张，标注 {len(lbls_a)} 个')
print(f'测试集 (B面)：图像 {len(imgs_b)} 张，标注 {len(lbls_b)} 个')
print(f'图像-标注匹配：{"✓" if len(imgs_a)==len(lbls_a) else "✗"}')

In [ ]:
# 读取标注文件，统计每张图片的种子数量
def count_seeds(label_path):
    """读取标注文件，返回种子数量"""
    try:
        with open(label_path, 'r') as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]
        return len(lines)
    except:
        return 0

def load_points(label_path):
    """读取标注文件，返回点坐标列表 [(x, y), ...]"""
    points = []
    try:
        with open(label_path, 'r') as f:
            for line in f:
                line = line.strip()
                if line:
                    coords = line.split()
                    if len(coords) >= 2:
                        points.append((float(coords[0]), float(coords[1])))
    except:
        pass
    return points

# 统计A面和B面每张图的种子数
counts_a = [count_seeds(p) for p in lbls_a]
counts_b = [count_seeds(p) for p in lbls_b]

print(f'A面种子数：均值={np.mean(counts_a):.1f}, 最小={min(counts_a)}, 最大={max(counts_a)}')
print(f'B面种子数：均值={np.mean(counts_b):.1f}, 最小={min(counts_b)}, 最大={max(counts_b)}')

## 4. 种子数量分布可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 直方图
for ax, counts, label, color in zip(
    axes,
    [counts_a, counts_b],
    ['A面（训练集）', 'B面（测试集）'],
    ['steelblue', 'coral']
):
    ax.hist(counts, bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(np.mean(counts), color='black', linestyle='--', linewidth=1.5,
               label=f'均值 = {np.mean(counts):.1f}')
    ax.set_xlabel('每张图片的种子数量', fontsize=12)
    ax.set_ylabel('图片数量', fontsize=12)
    ax.set_title(f'{label} 种子数量分布', fontsize=13)
    ax.legend(fontsize=11)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
os.makedirs('../outputs/figures', exist_ok=True)
plt.savefig('../outputs/figures/seed_count_distribution.png', bbox_inches='tight')
plt.show()
print('图表已保存至 outputs/figures/seed_count_distribution.png')

## 5. 图像与点标注可视化

可视化图像中每个大豆种子的点标注位置，直观展示标注质量。

In [ ]:
def visualize_sample(img_path, lbl_path, ax, title=None):
    """在图像上绘制点标注"""
    img = Image.open(img_path).convert('RGB')
    points = load_points(lbl_path)
    
    ax.imshow(img)
    if points:
        xs = [p[0] for p in points]
        ys = [p[1] for p in points]
        ax.scatter(xs, ys, c='red', s=15, alpha=0.8, linewidths=0.5,
                   edgecolors='white', zorder=5)
    
    ax.set_title(f'{title or img_path.stem}\n种子数：{len(points)}',
                 fontsize=10, pad=6)
    ax.axis('off')

# 随机抽取 6 张图像展示
np.random.seed(42)
sample_idx = np.random.choice(len(imgs_a), size=min(6, len(imgs_a)), replace=False)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('训练集（A面）样本图像与点标注可视化', fontsize=14, y=1.01)

for i, idx in enumerate(sample_idx):
    ax = axes[i // 3][i % 3]
    visualize_sample(imgs_a[idx], lbls_a[idx], ax)

plt.tight_layout()
plt.savefig('../outputs/figures/sample_annotations.png', bbox_inches='tight')
plt.show()
print('图表已保存至 outputs/figures/sample_annotations.png')

## 6. 图像尺寸统计

In [ ]:
# 抽样统计图像尺寸（避免全部读取耗时过长）
sample_imgs = np.random.choice(imgs_a, size=min(50, len(imgs_a)), replace=False)
sizes = [Image.open(p).size for p in sample_imgs]
widths  = [s[0] for s in sizes]
heights = [s[1] for s in sizes]

print(f'图像宽度：{min(widths)} ~ {max(widths)} px（均值 {np.mean(widths):.0f}）')
print(f'图像高度：{min(heights)} ~ {max(heights)} px（均值 {np.mean(heights):.0f}）')

# 统计唯一尺寸
unique_sizes = set(sizes)
print(f'\n共 {len(unique_sizes)} 种不同尺寸：')
for s in sorted(unique_sizes):
    print(f'  {s[0]} × {s[1]}')

## 7. A面 vs B面 种子数量对比

论文核心实验设计：A面训练 → B面测试，验证模型在不同拍摄视角下的泛化能力。

In [ ]:
# 构建 DataFrame 用于汇总分析
df_a = pd.DataFrame({
    'filename': [p.stem for p in lbls_a],
    'count': counts_a,
    'split': 'A面（训练）'
})
df_b = pd.DataFrame({
    'filename': [p.stem for p in lbls_b],
    'count': counts_b,
    'split': 'B面（测试）'
})

df_all = pd.concat([df_a, df_b], ignore_index=True)

# 箱线图对比
fig, ax = plt.subplots(figsize=(8, 5))
groups = [df_a['count'].values, df_b['count'].values]
bp = ax.boxplot(groups, labels=['A面（训练集）', 'B面（测试集）'],
                patch_artist=True, notch=False)

colors = ['steelblue', 'coral']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('每张图片的种子数量', fontsize=12)
ax.set_title('训练集与测试集种子数量分布对比', fontsize=13)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/figures/ab_count_comparison.png', bbox_inches='tight')
plt.show()

print('\n数据集统计摘要：')
print(df_all.groupby('split')['count'].describe().round(1))

## 8. 本章小结

通过本 Notebook 的数据探索，我们了解到：

- **数据规模**：训练集（A面）和测试集（B面）各包含若干裁剪图像块，每个图像块对应一份点标注文件
- **标注方式**：每个大豆种子用一个点坐标表示，标注文件为纯文本格式（每行 `x y`）
- **种子密度**：不同品系、不同图像块中的种子数量差异较大，体现了计数任务的挑战性
- **实验设计**：A面与B面图像来自同一植株的两侧拍摄，模型需在视角变化下保持准确性

下一步：在 `02_model_inference.ipynb` 中加载预训练模型并进行推理预测。